# Install and Import Miletos

This notebook reproduces the four-planet transit analysis of HD 108236 (TOI-1233) from Daylan et al. (2021), *AJ*, 161, 85, DOI `10.3847/1538-3881/abd73e`. Run the notebook from the Miletos repository root. If the package is not installed, run `%pip install -e .` once before the import below.

In [ ]:
from importlib.metadata import version

from miletos import main
from miletos.paths import get_repository_path

print(f"Miletos {version('miletos')}")

# Configure the Daylan et al. (2021) Example

The configuration fixes the analysis to the two publication-era Transiting Exoplanet Survey Satellite (TESS) sectors and the four planets reported in Tables 11 and 12. Miletos discovers, retrieves, quality-masks, normalizes, and formats the Science Processing Operations Center presearch data conditioning (SPOC PDC) light curves.

In [ ]:
target = {
    "ticitarg": 260647166,
    "strgtarg": "TOI-1233",
    "labltarg": "HD 108236",
}

published_priors = {
    "liststrgcomp": ["b", "c", "d", "e"],
    "rratcompprio": [0.01638, 0.02134, 0.02805, 0.0323],
    "rsmacompprio": [0.0895, 0.0647, 0.0375, 0.03043],
    "epocmtracompprio": [2458572.1128, 2458572.3949, 2458571.3368, 2458586.5677],  # [BJD]
    "pericompprio": [3.79523, 6.20370, 14.17555, 19.5917],  # [day]
    "cosicompprio": [0.037, 0.022, 0.0136, 0.0118],
    "stdvepocmtracompprio": [0.00335, 0.00225, 0.00140, 0.00140],  # [day]
    "stdvpericompprio": [0.000455, 0.000580, 0.001045, 0.00210],  # [day]
}

output_path = get_repository_path() / "examples" / "TOI-1233"

# Run the Automated Miletos Pipeline

This call delegates data discovery, download, quality filtering, normalization, Gaussian-process baseline modeling, transit fitting, and standard figure generation to Miletos. No observational arrays are loaded or transformed in the notebook.

In [ ]:
result = main.init(
    **target,
    **published_priors,
    pathtarg=f"{output_path}/",
    strgcnfg="Daylan2021",
    listtsecsele=[10, 11],
    typelcurtpxftess="SPOC",
    typedataspoc="PDC",
    dictfitt={"typemodl": "PlanetarySystem"},
    typepriocomp="inpt",
    typebdtr="GaussianProcess",
    boolanls=True,
    boolfitt=True,
    typeinfe="samp",
    boolplot=True,
    boolplotpopl=False,
    typefileplot="png",
    typeplotback="white",
)

# Inspect Pipeline Products

Miletos returns a structured dictionary containing target metadata and generated analysis products. The pipeline also retains the processed time series, fitted parameters, and fit diagnostics in its standard target output directories.

In [ ]:
print("Returned Miletos products:")
for name in sorted(result):
    print(f"  {name}: {type(result[name]).__name__}")

# Visualize the Results

The pipeline writes its standard observation, fitted-model, residual, and phase-folded figures. This section displays those Miletos-generated products without reconstructing plots in the notebook.

In [ ]:
from IPython.display import Image, display

visual_path = output_path / "Daylan2021" / "visuals"
figure_paths = sorted(visual_path.glob("*.png"))
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))

# Validate and Save Outputs

Miletos saves figures and machine-readable fitting products during the pipeline call. The checks below confirm completion and the presence of both output classes without rewriting any pipeline product.

In [ ]:
run_path = output_path / "Daylan2021"
machine_readable_paths = sorted(run_path.rglob("*.csv"))

assert result["strgtarg"] == "TOI-1233"
assert figure_paths, "Miletos did not produce PNG figures."
assert machine_readable_paths, "Miletos did not produce machine-readable outputs."

print(f"Validated {len(figure_paths)} figures in {visual_path}.")
print(f"Validated {len(machine_readable_paths)} machine-readable products in {run_path}.")